# 01. Data Preparation

This notebook reads the raw sources, cleans them, and saves small ready-to-map files in `data/processed`. The map itself is built in a separate notebook (`02buildmap.ipynb`), which only reads those small files.

| Section | What it does |
|---|---|
| 0 | Settings: the period, top N destinations, folders |
| 1 | Load Rapid Rail ridership (downloaded once, then cached) |
| 2 | Split the data into station totals and station-to-station flows |
| 3 | Station table: average daily entries, working day vs weekend and holiday |
| 4 | Flow table: each station's top destinations |
| 5 | Station locations from GTFS, matched to the ridership stations |
| 6 | Klang Valley boundaries: districts and parliamentary constituencies |
| 7 | Save the outputs |
| 8 | How to refresh with newer data |

**Libraries:** pandas, pyarrow, geopandas, requests and holidays. If `holidays` is missing, run this once in the VS Code terminal from the KL1 folder: `uv pip install --python .venv holidays`

## 0. Settings

In [1]:
from pathlib import Path
import io, re, json, zipfile
from datetime import datetime
import requests
import pandas as pd
import geopandas as gpd
import holidays

# ---- Period to analyse (inclusive) ----
START_DATE = "2026-06-01"
END_DATE   = "2026-08-31"

# ---- Map settings ----
TOP_N = 5                  # destinations kept per station

# ---- Downloads ----
REFRESH_DOWNLOAD = False   # True = download fresh copies even if cached files exist

# ---- Area ----
KV_DISTRICTS = ["W.P. Kuala Lumpur", "W.P. Putrajaya",
                "Petaling", "Gombak", "Ulu Langat", "Klang", "Sepang"]

# ---- Folders ----
RAW = Path("data/raw")
PROCESSED = Path("data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# ---- Source addresses ----
OD_URL   = "https://storage.data.gov.my/transportation/rail/rapidrail_{year}_daily.parquet"
GTFS_URL = "https://api.data.gov.my/gtfs-static/prasarana?category=rapid-rail-kl"
GEO_URL  = "https://raw.githubusercontent.com/dosm-malaysia/data-open/main/datasets/geodata"

start, end = pd.Timestamp(START_DATE), pd.Timestamp(END_DATE)
print(f"Period: {start.date()} to {end.date()} ({(end - start).days + 1} days) | top {TOP_N} destinations")

Period: 2026-06-01 to 2026-08-31 (92 days) | top 5 destinations


## 1. Load Rapid Rail ridership

data.gov.my publishes one file per year. The function below downloads a year's file the first time and saves it in `data/raw`, so later runs read it from disk in seconds. If the period crosses a year boundary, both years are loaded automatically.

In [2]:
def get_file(url, path):
    """Download url to path, unless a cached copy exists and REFRESH_DOWNLOAD is False."""
    if REFRESH_DOWNLOAD or not path.exists():
        print(f"Downloading {url}")
        r = requests.get(url, timeout=600)
        r.raise_for_status()
        path.write_bytes(r.content)
    else:
        print(f"Using cached {path} (saved {datetime.fromtimestamp(path.stat().st_mtime):%Y-%m-%d})")
    return path

years = range(start.year, end.year + 1)
od = pd.concat(
    [pd.read_parquet(get_file(OD_URL.format(year=y), RAW / f"rapidrail_{y}_daily.parquet")) for y in years],
    ignore_index=True)
od["date"] = pd.to_datetime(od["date"])
print(f"Loaded {len(od):,} rows covering {od['date'].min().date()} to {od['date'].max().date()}")

od = od[(od["date"] >= start) & (od["date"] <= end)].copy()
days_present = od["date"].nunique()
days_expected = (end - start).days + 1
print(f"Kept {len(od):,} rows for the period: {days_present} of {days_expected} days present")
if days_present < days_expected:
    print("WARNING: some days are missing. If END_DATE is recent, set REFRESH_DOWNLOAD = True and rerun.")

Using cached data\raw\rapidrail_2026_daily.parquet (saved 2026-09-28)
Loaded 4,685,040 rows covering 2026-01-01 to 2026-09-27
Kept 1,596,384 rows for the period: 92 of 92 days present


## 2. Split into totals and flows

The file mixes two kinds of rows (found in the smoke test):

- **Station totals:** a row from a station to `A0: All Stations` is that station's total entries for the day; a row from `A0` to a station is its total exits.
- **Flows:** rows between two real stations.

Mixing them doubles every count, so they are separated here.

**Known gap in the source data (found 28 September 2026):** the 32 Putrajaya Line stations from Damansara Damai (PYL05) to Putrajaya Sentral (PYL41) record entries, but no journey is ever recorded as ending there. Their exits are credited to other stations across the network, so the true destination cannot be recovered. For that reason this notebook measures station busyness by **entries only**, which are recorded correctly everywhere. The check below lists any station with entries but no exits, so you can see whether the gap is still there after a refresh.

In [3]:
is_a0_origin = od["origin"].str.startswith("A0:")
is_a0_dest   = od["destination"].str.startswith("A0:")

entries = od[~is_a0_origin &  is_a0_dest][["origin", "date", "ridership"]].rename(columns={"origin": "od_station"})
exits   = od[ is_a0_origin & ~is_a0_dest][["destination", "date", "ridership"]].rename(columns={"destination": "od_station"})
flows   = od[~is_a0_origin & ~is_a0_dest].copy()

print(f"Entry rows: {len(entries):,} | exit rows: {len(exits):,} | flow rows: {len(flows):,}")

# Check: a station's flows should add up to roughly its entry total
check = (flows.groupby("origin")["ridership"].sum()
         / entries.groupby("od_station")["ridership"].sum()).dropna()
no_exits = sorted(set(entries["od_station"]) - set(exits["od_station"]))
print(f"Stations with entries but no exit rows: {len(no_exits)}"
      + (f" ({no_exits[0]} to {no_exits[-1]})" if no_exits else ""))

print(f"Flows as a share of entry totals: median {check.median():.1%} "
      f"(range {check.min():.1%} to {check.max():.1%}); close to 100% means the split is right")

Entry rows: 15,272 | exit rows: 12,328 | flow rows: 1,568,784
Stations with entries but no exit rows: 32 (PYL05: Damansara Damai to PYL41: Putrajaya Sentral)
Flows as a share of entry totals: median 100.0% (range 100.0% to 100.0%); close to 100% means the split is right


## 3. Station table

Stations are grouped by **cleaned name**, so an interchange such as Titiwangsa (four lines) becomes one place on the map. Level of activity is measured as **average daily entries** (people tapping in), because exits are incomplete in the source (see Section 2).

**New stations:** a station that opened during the period is averaged only over the days it has data, not the whole period, so it is not understated. For example, the Shah Alam Line (LRT3) opened on 29 June 2026 with free rides until 31 July, so its stations have taps only from 1 August. Each line code is averaged over its own days, and a place's figures are the sum of its line codes. Places with a shorter record get a `service_note`, which the map shows.

Days are split into **working days** and **weekends and public holidays**, using the Kuala Lumpur holiday calendar. Holidays behave like weekends for travel, so counting them as working days would pull those averages down.

`weekend_ratio` = average weekend and holiday footfall divided by average working-day footfall. Around 0.3 suggests a commuter or office station; near 1 or above suggests shopping, leisure or residential use.

In [4]:
def clean(s):
    """Station name to a matching key: 'KG18: Bukit Bintang' -> 'bukitbintang'."""
    s = str(s).lower()
    s = re.sub(r"^[a-z]{1,3}\d{1,3}\s*[:\-]\s*", "", s)   # leading line code, e.g. 'KG18: ' or 'PYL17: '
    s = s.split(" - ")[0]                                   # sponsor suffix, e.g. 'KL SENTRAL - REDONE'
    s = re.sub(r"\b(lrt|mrt|monorail|stesen|station)\b", "", s)
    return re.sub(r"[^a-z0-9]", "", s)

def norm_code(s):
    """Line code with leading zeros removed: 'AG09: Bandaraya' -> 'AG9'."""
    m = re.match(r"^\s*([A-Za-z]{1,3})0*(\d{1,3})\b", str(s))
    return (m.group(1) + m.group(2)).upper() if m else None

# Day types
kl_holidays = holidays.Malaysia(years=list(years), subdiv="KUL")
all_days = pd.DataFrame({"date": pd.date_range(start, end)})
all_days["is_holiday"] = all_days["date"].dt.date.map(lambda d: d in kl_holidays)
all_days["day_type"] = "working"
all_days.loc[(all_days["date"].dt.dayofweek >= 5) | all_days["is_holiday"], "day_type"] = "weekend_holiday"
print("Days by type:", all_days["day_type"].value_counts().to_dict())
print("Public holidays in period:",
      [f"{d.date()} {kl_holidays.get(d.date())}" for d in all_days.loc[all_days['is_holiday'], 'date']])

# Daily entries per line code (e.g. 'SA01: Bandar Utama'), one column per day in the period
entries["key"] = entries["od_station"].map(clean)
daily = entries.groupby(["od_station", "date"])["ridership"].sum().unstack(fill_value=0)
daily = daily.reindex(columns=all_days["date"], fill_value=0)

# Days in service: from each code's first day with entries. Earlier days are left out (not counted as zero),
# so a station that opened during the period is averaged over the days it actually had data.
first_day = daily.gt(0).idxmax(axis=1).where(daily.gt(0).any(axis=1))
in_service = pd.DataFrame({d: first_day <= d for d in daily.columns})
daily = daily.where(in_service)
service_days = daily.notna().sum(axis=1)

working = all_days.loc[all_days["day_type"] == "working", "date"]
weekend = all_days.loc[all_days["day_type"] == "weekend_holiday", "date"]

code_stats = pd.DataFrame({
    "avg_daily_entries":   daily.mean(axis=1),
    "avg_working_day":     daily[working].mean(axis=1),
    "avg_weekend_holiday": daily[weekend].mean(axis=1),
    "service_days":        service_days,
    "first_day":           first_day,
}).dropna(subset=["avg_daily_entries"])
code_stats["key"] = code_stats.index.map(clean)

# A place's figures are the sum of its line codes
stations = code_stats.groupby("key")[["avg_daily_entries", "avg_working_day", "avg_weekend_holiday"]].sum()
stations["service_days"] = code_stats.groupby("key")["service_days"].max()
stations["weekend_ratio"] = stations["avg_weekend_holiday"] / stations["avg_working_day"]

# Note for places where any line code has a shorter record than the full period
short = code_stats[code_stats["service_days"] < len(all_days)]
notes = short.groupby("key").apply(
    lambda g: "; ".join(f"{c.split(':')[0]} data from {d:%d %b %Y} ({n} days)"
                        for c, d, n in zip(g.index, g["first_day"], g["service_days"])))
stations["service_note"] = stations.index.map(notes).fillna("")
print(f"Line codes with a shorter record than the {len(all_days)}-day period: {len(short)}")
if len(short):
    display(short[["first_day", "service_days", "avg_daily_entries"]].sort_values("first_day"))

# Display name and line codes for each place
names = entries.drop_duplicates("od_station").copy()
names["name"] = names["od_station"].str.replace(r"^[A-Za-z]{1,3}\d{1,3}\s*:\s*", "", regex=True)
names["code"] = names["od_station"].map(norm_code)
stations = stations.join(names.groupby("key").agg(name=("name", "first"), codes=("code", lambda c: ", ".join(sorted(set(c))))))
stations = stations.round({"avg_daily_entries": 0, "avg_working_day": 0, "avg_weekend_holiday": 0, "weekend_ratio": 3})
stations.index.name = "key"

print(f"\n{len(stations)} places from {entries['od_station'].nunique()} station codes")
display(stations.sort_values("avg_daily_entries", ascending=False).head(8))
print("Most commuter-like (lowest weekend_ratio):")
display(stations.nsmallest(5, "weekend_ratio")[["name", "avg_working_day", "avg_weekend_holiday", "weekend_ratio"]])
print("Most weekend-heavy (highest weekend_ratio):")
display(stations.nlargest(5, "weekend_ratio")[["name", "avg_working_day", "avg_weekend_holiday", "weekend_ratio"]])

Days by type: {'working': 61, 'weekend_holiday': 31}
Public holidays in period: ['2026-06-01 Hari Keputeraan Rasmi Seri Paduka Baginda Yang di-Pertuan Agong', '2026-06-02 Cuti Hari Wesak', '2026-06-17 Awal Muharam', '2026-08-25 Hari Keputeraan Nabi Muhammad S.A.W.', '2026-08-31 Hari Kebangsaan']
Line codes with a shorter record than the 92-day period: 20


,first_day,service_days,avg_daily_entries
od_station,,,
SA01: Bandar Utama,2026-08-01,31,5927.903226
SA23: Klang Jaya,2026-08-01,31,512.903226
SA22: Seri Andalas,2026-08-01,31,982.064516
SA21: Taman Selatan,2026-08-01,31,520.838710
SA20: Jambatan Kota,2026-08-01,31,1383.806452
SA19: Jalan Meru,2026-08-01,31,363.096774
SA18: Pasar Klang,2026-08-01,31,1431.258065
SA17: Bandar Baru Klang,2026-08-01,31,716.193548
SA15: Seksyen 7,2026-08-01,31,1736.580645



154 places from 166 station codes


,avg_daily_entries,avg_working_day,avg_weekend_holiday,service_days,weekend_ratio,service_note,name,codes
key,,,,,,,,
bukitbintang,35031.0,34965.0,35160.0,92,1.006,,Bukit Bintang,"KG18, MR6"
klsentral,26398.0,29455.0,20383.0,92,0.692,,KL Sentral,"KJ15, MR1"
klcc,21710.0,24492.0,16235.0,92,0.663,,KLCC,KJ10
pasarseni,19506.0,17881.0,22704.0,92,1.270,,Pasar Seni,"KG16, KJ14"
ampangpark,19078.0,23758.0,9868.0,92,0.415,,Ampang Park,"KJ9, PYL20"
bandarutama,17467.0,19274.0,14170.0,92,0.735,SA01 data from 01 Aug 2026 (31 days),Bandar Utama,"KG9, SA1"
maluri,17395.0,18654.0,14916.0,92,0.800,,Maluri,"AG13, KG22"
masjidjamek,15786.0,16911.0,13575.0,92,0.803,,Masjid Jamek,"AG7, KJ13"


Most commuter-like (lowest weekend_ratio):


,name,avg_working_day,avg_weekend_holiday,weekend_ratio
key,,,,
kerinchi,Kerinchi,8703.0,2314.0,0.266
semantan,Semantan,6772.0,2189.0,0.323
rajachulan,Raja Chulan,5148.0,1729.0,0.336
sultanismail,Sultan Ismail,2241.0,770.0,0.344
asiajaya,Asia Jaya,6505.0,2380.0,0.366


Most weekend-heavy (highest weekend_ratio):


,name,avg_working_day,avg_weekend_holiday,weekend_ratio
key,,,,
pasarseni,Pasar Seni,17881.0,22704.0,1.270
imbi,Imbi,4823.0,5632.0,1.168
hangtuah,Hang Tuah,9620.0,10560.0,1.098
cochrane,Cochrane,8818.0,9660.0,1.095
chowkit,Chow Kit,3488.0,3779.0,1.084


## 4. Flow table: top destinations

Average daily trips between each pair of places over the whole period. Trips that start and end at the same place are dropped. Then only the `TOP_N` busiest destinations per origin are kept.

Because of the Putrajaya Line gap (Section 2), no flow ends at a Putrajaya Line station from Damansara Damai onwards. Trips from those stations are still counted.

In [5]:
flows["origin_key"] = flows["origin"].map(clean)
flows["dest_key"]   = flows["destination"].map(clean)
flows = flows[flows["origin_key"] != flows["dest_key"]]

# Average over the origin line code's days in service (see Section 3), then add up line codes per place
by_code = flows.groupby(["origin", "origin_key", "dest_key"])["ridership"].sum().reset_index()
by_code["avg_daily_trips"] = by_code["ridership"] / by_code["origin"].map(service_days)
pair = by_code.groupby(["origin_key", "dest_key"])["avg_daily_trips"].sum().reset_index()
pair["share_of_origin"] = pair["avg_daily_trips"] / pair.groupby("origin_key")["avg_daily_trips"].transform("sum")
pair["rank"] = pair.groupby("origin_key")["avg_daily_trips"].rank(method="first", ascending=False).astype(int)

top = pair[pair["rank"] <= TOP_N].sort_values(["origin_key", "rank"]).reset_index(drop=True)
top["avg_daily_trips"] = top["avg_daily_trips"].round(0)
top["share_of_origin"] = top["share_of_origin"].round(3)

print(f"{len(pair):,} place pairs with traffic; kept {len(top):,} rows (top {TOP_N} for each of {top['origin_key'].nunique()} origins)")
print(f"On average the top {TOP_N} carry {top.groupby('origin_key')['share_of_origin'].sum().mean():.0%} of a station's trips")
display(top[top["origin_key"] == stations["avg_daily_entries"].idxmax()])

14,726 place pairs with traffic; kept 770 rows (top 5 for each of 154 origins)
On average the top 5 carry 43% of a station's trips


,origin_key,dest_key,avg_daily_trips,share_of_origin,rank
100,bukitbintang,maluri,2230.0,0.066,1
101,bukitbintang,pasarseni,2086.0,0.062,2
102,bukitbintang,tunrazakexchange,1878.0,0.056,3
103,bukitbintang,bukitnanas,1439.0,0.043,4
104,bukitbintang,cochrane,1389.0,0.041,5


## 5. Station locations (GTFS)

Coordinates come from `stops.txt` in the GTFS feed. Each place takes the average position of its GTFS stops, so interchanges and sponsor-named duplicates collapse into one point. Places are then tagged with their district and constituency.

In [6]:
gtfs_path = get_file(GTFS_URL, RAW / "gtfs_rapid_rail_kl.zip")
with zipfile.ZipFile(gtfs_path) as z:
    stops = pd.read_csv(z.open("stops.txt"))
stops["key"] = stops["stop_name"].map(clean)
stops["code"] = stops["stop_id"].map(norm_code)
print(f"{len(stops)} GTFS stops")

# Match each place by name first, then by any of its line codes
by_name = stops.groupby("key")[["stop_lat", "stop_lon"]].mean()
by_code = stops.groupby("code")[["stop_lat", "stop_lon"]].mean()

def locate(row):
    if row.name in by_name.index:
        return by_name.loc[row.name]
    for c in row["codes"].split(", "):
        if c in by_code.index:
            return by_code.loc[c]
    return pd.Series({"stop_lat": None, "stop_lon": None})

stations[["lat", "lon"]] = stations.apply(locate, axis=1).values.astype(float)
missing = stations[stations["lat"].isna()]
print(f"Located {stations['lat'].notna().sum()} of {len(stations)} places")
if len(missing):
    print("NOT LOCATED (add these by hand or check for renamed stations):")
    display(missing[["name", "codes"]])

stations_gdf = gpd.GeoDataFrame(
    stations.dropna(subset=["lat"]).reset_index(),
    geometry=gpd.points_from_xy(stations.dropna(subset=["lat"])["lon"], stations.dropna(subset=["lat"])["lat"]),
    crs="EPSG:4326")

Using cached data\raw\gtfs_rapid_rail_kl.zip (saved 2026-09-28)
187 GTFS stops
Located 154 of 154 places


## 6. Klang Valley boundaries

Districts come from DOSM's administrative file. Constituencies are kept if they sit inside one of those districts. Stations are tagged with their district and constituency using the exact shapes; the shapes are then simplified slightly (about 50 m) so the web page loads quickly.

In [7]:
districts_all = gpd.read_file(f"{GEO_URL}/administrative_2_district.geojson")
parlimen_all  = gpd.read_file(f"{GEO_URL}/electoral_0_parlimen.geojson")

districts = districts_all[districts_all["district"].isin(KV_DISTRICTS)][["state", "district", "geometry"]].reset_index(drop=True)
missing_d = set(KV_DISTRICTS) - set(districts["district"])
print(f"{len(districts)} districts" + (f"; NOT FOUND: {missing_d}" if missing_d else ""))

# A constituency belongs to the Klang Valley if its centre point lies in one of the districts
centres = parlimen_all.copy()
centres["geometry"] = parlimen_all.geometry.representative_point()
inside = gpd.sjoin(centres, districts[["district", "geometry"]], predicate="within")
constituencies = parlimen_all.loc[inside.index, ["state", "parlimen", "geometry"]].copy()
constituencies["district"] = inside["district"].values
constituencies = constituencies.reset_index(drop=True)
print(f"{len(constituencies)} parliamentary constituencies in the Klang Valley")

# Tag each station with its district and constituency
stations_gdf = gpd.sjoin(stations_gdf, districts[["district", "geometry"]], how="left", predicate="within").drop(columns="index_right")
stations_gdf = gpd.sjoin(stations_gdf, constituencies[["parlimen", "geometry"]], how="left", predicate="within").drop(columns="index_right")
# A station exactly on a boundary can match two shapes; keep one row per station
stations_gdf = stations_gdf.drop_duplicates("key").reset_index(drop=True)

# Simplify the shapes only after tagging, so tagging uses the exact boundaries
for g in (districts, constituencies):
    g["geometry"] = g.geometry.simplify(0.0005, preserve_topology=True)

outside = stations_gdf["district"].isna().sum()
print(f"{len(stations_gdf)} stations tagged")
print(f"Stations outside the {len(KV_DISTRICTS)} districts: {outside}")
print(stations_gdf["district"].value_counts().to_string())

7 districts
27 parliamentary constituencies in the Klang Valley
154 stations tagged
Stations outside the 7 districts: 0
district
W.P. Kuala Lumpur    75
Petaling             53
Ulu Langat           12
Klang                 9
Sepang                3
W.P. Putrajaya        1
Gombak                1


## 7. Save the outputs

In [8]:
stations_gdf.to_file(PROCESSED / "stations.geojson", driver="GeoJSON")
name_of = stations["name"].to_dict()
top.insert(1, "origin_name", top["origin_key"].map(name_of))
top.insert(3, "dest_name", top["dest_key"].map(name_of))
top.to_csv(PROCESSED / "flows_top.csv", index=False)

districts.to_file(PROCESSED / "districts.geojson", driver="GeoJSON")
constituencies.to_file(PROCESSED / "constituencies.geojson", driver="GeoJSON")

meta = {
    "period_start": START_DATE, "period_end": END_DATE,
    "days": len(all_days), "working_days": int((all_days["day_type"] == "working").sum()),
    "weekend_holiday_days": int((all_days["day_type"] == "weekend_holiday").sum()),
    "top_n": TOP_N, "prepared_on": datetime.now().strftime("%Y-%m-%d"),
    "busyness_measure": "average daily entries (tap-ins)",
    "data_notes": ["Exits at 32 Putrajaya Line stations (Damansara Damai to Putrajaya Sentral) are not recorded "
                   "in the source; those trips are credited to other stations. Busyness therefore uses entries only, "
                   "and no flow ends at those stations.",
                   "Stations that opened during the period are averaged over the days they have data: the Shah Alam "
                   "Line (LRT3) opened on 29 June 2026 with free rides until 31 July, so its figures cover August only."],
    "sources": {"ridership": "data.gov.my, Daily Origin-Destination Ridership: Rapid Rail (KV), CC BY 4.0",
                "stations": "data.gov.my, GTFS Static (Prasarana rapid-rail-kl), CC BY 4.0",
                "boundaries": "Department of Statistics Malaysia (DOSM), data-open geodata"},
}
(PROCESSED / "metadata.json").write_text(json.dumps(meta, indent=2))

for f in sorted(PROCESSED.iterdir()):
    print(f"{f.name:25} {f.stat().st_size / 1e3:8.0f} KB")

constituencies.geojson          56 KB
context_metadata.json            1 KB
districts.geojson               30 KB
districts_income.geojson        30 KB
flows_top.csv                   49 KB
landuse_near_stations.geojson      557 KB
metadata.json                    1 KB
population_metadata.json         1 KB
station_buildings.csv            6 KB
station_context.csv             11 KB
station_population.csv           9 KB
stations.geojson                66 KB


## 8. How to refresh with newer data

The whole notebook is driven by the settings in Section 0, so a refresh is mostly a change of dates.

1. **Change the period** in Section 0, e.g. `START_DATE = "2026-09-01"` and `END_DATE = "2026-11-30"`. Keep it to about three full months, and note any festive season inside it (Hari Raya, Chinese New Year, Deepavali, school holidays), because those change travel patterns.
2. **Set `REFRESH_DOWNLOAD = True`** for that run. The current year's file on data.gov.my grows every day, so the cached copy in `data/raw` becomes out of date. This also fetches the latest station list. Set it back to `False` afterwards.
3. **Run all cells**, then check the warnings:
    - Section 1 warns if days are missing (the source is usually one or two days behind).
    - Section 5 lists any station it could not place. New or renamed stations appear here and may need a small fix to the name matching.
4. **Rerun the map notebook** (`02_build_v1_map.ipynb`). It reads the new files, and the period shown on the map comes from `metadata.json`.
5. **Republish** Updates the map.

A period that crosses New Year (e.g. November to January) works without changes: both yearly files are loaded automatically. Old raw files can be deleted from `data/raw` to save space; they are about 30 MB per year.

___
**END**